In [ ]:
%%sql
SELECT DISTINCT
    e.is_dna,
    s.description AS activity_status_description,
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END AS derived_session_status_src_name
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activityheader h
    ON e.activity_header_id = h.id
LEFT JOIN silver_wip_activitystatus s
    ON h.activity_status_id = s.id
ORDER BY 1,2,3;

In [ ]:
%%sql
SELECT
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END AS derived_session_status_src_name,
    COUNT(*) AS cnt
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activityheader h
    ON e.activity_header_id = h.id
LEFT JOIN silver_wip_activitystatus s
    ON h.activity_status_id = s.id
GROUP BY
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(TRIM(s.description)) LIKE '%cancel%' THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
    END
ORDER BY cnt DESC;

In [ ]:
,
wip_source AS (
    -- WIP source values derived from activity entry, activity header and activity status
    -- current available WIP logic supports Attended and Did Not Attend for WIP001
    SELECT DISTINCT
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END AS session_status_src_name,

        LOWER(TRIM(
            CASE
                WHEN e.is_dna = true THEN 'Did Not Attend'
                WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
            END
        )) AS session_status_src_id,

        'WIP001' AS session_status_src_sys_inst_id
    FROM silver_wip_activityentry e
    LEFT JOIN silver_wip_activityheader h
        ON e.activity_header_id = h.id
    LEFT JOIN silver_wip_activitystatus st
        ON h.activity_status_id = st.id
    WHERE CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
          END IS NOT NULL
)

In [ ]:
FROM (
    SELECT * FROM mpb_source
    UNION
    SELECT * FROM wip_source
) s